# Retrieval models on the full CF collection

Run this notebook from top to bottom to preprocess all **1,239 documents**, fit all **100 queries**, and inspect retrieval results.

Models: **SetBasedModel**, **GSBModel**, and **WindowedGSBModel**, including k-core weighting and fixed/fractional window configurations.

Documents are lowercased and split on punctuation (numbers kept, no stemming); queries get the same treatment automatically, plus optional stopword removal via `stopwords=True` when fitting. Documents themselves keep their stopwords. Change the settings below to explore different runs.

In [1]:
from pathlib import Path
import sys
from time import perf_counter

from sympy import true

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/graph_ir").is_dir() and (path / "pyproject.toml").is_file()
)
sys.dont_write_bytecode = True
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import numpy as np
from IPython.display import Markdown, display

from graph_ir.data.collection import Collection
from graph_ir.data.preprocessing.text import preprocess_text, remove_stopwords
from graph_ir.models.setbased import SetBasedModel
from graph_ir.models.graphical_setbased import GSBModel
from graph_ir.models.windowed_graphical_setbased import WindowedGSBModel

MIN_FREQ = 1
FIXED_WINDOW = 8
FRACTIONAL_WINDOW = 0.1
CORE_WEIGHT = 2

def show_table(headers, rows):
    def row(values):
        return "| " + " | ".join(str(value).replace("|", "&#124;") for value in values) + " |"
    display(Markdown("\n".join([row(headers), row(["---"] * len(headers)), *map(row, rows)])))

print("Collection path:", ROOT / "collections/cf")

Collection path: /home/nikitas/PycharmProjects/Graph_ir/collections/cf


## Load and preprocess the entire collection

The collection's preprocessor is also applied to queries by `model.fit()`. Load every document and query; no subsets are selected.

In [2]:
collection = Collection(ROOT / "collections/cf")
preprocessor = preprocess_text

collection.create(preprocessor=preprocessor).load_queries().load_qrels()

print(f"Documents: {collection.num_docs}")
print(f"Queries: {len(collection.queries)}")
print(f"Vocabulary: {len(collection.inverted_index)} terms")
print(f"Relevance judgments: {sum(map(len, collection.qrels.values()))}")
print("Example preprocessing:", preprocessor("THE CF patients, patients! IL-6"))

sample_query_id = next(iter(collection.queries))
print("Sample query:", collection.queries[sample_query_id])
print("Query tokens:", preprocessor(collection.queries[sample_query_id]))

Documents: 1239
Queries: 100
Vocabulary: 10109 terms
Relevance judgments: 4819
Example preprocessing: ['the', 'cf', 'patients', 'patients', 'il', '6']
Sample query: What are the effects of calcium on the physical properties of mucus from CF patients?
Query tokens: ['what', 'are', 'the', 'effects', 'of', 'calcium', 'on', 'the', 'physical', 'properties', 'of', 'mucus', 'from', 'cf', 'patients']


## Create the models

`SetBasedModel` uses IDF-weighted frequent termsets. `GSBModel` adds graph-derived term weights using whole-document co-occurrence. `WindowedGSBModel` uses consecutive, non-overlapping token windows.

The k-core variants use `h=2` to amplify contributions from maximum-core terms. Fractional windows use 10% of each preprocessed document's length. Both windowed configurations enforce a seven-token minimum and retain the last partial window.

All models can share this collection because the current graph models own their node weights in `model._nwk`.

In [3]:
model_specs = [
    ("SetBasedModel", SetBasedModel, {}),
    ("GSBModel", GSBModel, {}),
    ("GSBModel + k-core", GSBModel, {"k_core_bool": True, "h_val": CORE_WEIGHT}),
    ("WindowedGSBModel: fixed", WindowedGSBModel, {"window": FIXED_WINDOW}),
    ("WindowedGSBModel: fixed + k-core", WindowedGSBModel,
     {"window": FIXED_WINDOW, "k_core_bool": True, "h_val": CORE_WEIGHT}),
    ("WindowedGSBModel: fractional", WindowedGSBModel, {"window": FRACTIONAL_WINDOW}),
    ("WindowedGSBModel: fractional + k-core", WindowedGSBModel,
     {"window": FRACTIONAL_WINDOW, "k_core_bool": True, "h_val": CORE_WEIGHT}),
]

models = {}
timings = {}
graph_rows = []
for name, model_class, parameters in model_specs:
    start = perf_counter()
    model = model_class(collection, **parameters)
    timings[name] = {"build": perf_counter() - start}
    models[name] = model
    graph = getattr(model, "graph", None)
    graph_rows.append((name, len(graph) if graph is not None else "—",
                       graph.number_of_edges() if graph is not None else "—",
                       f"{timings[name]['build']:.2f}"))
    print(f"Created {name}", flush=True)

show_table(("Model", "Graph nodes", "Graph edges", "Build time (s)"), graph_rows)

Created SetBasedModel
Created GSBModel
Created GSBModel + k-core
Created WindowedGSBModel: fixed
Created WindowedGSBModel: fixed + k-core
Created WindowedGSBModel: fractional
Created WindowedGSBModel: fractional + k-core


| Model | Graph nodes | Graph edges | Build time (s) |
| --- | --- | --- | --- |
| SetBasedModel | — | — | 0.00 |
| GSBModel | 10109 | 2064337 | 2.48 |
| GSBModel + k-core | 10109 | 2064337 | 2.41 |
| WindowedGSBModel: fixed | 10109 | 252852 | 1.21 |
| WindowedGSBModel: fixed + k-core | 10109 | 252852 | 1.93 |
| WindowedGSBModel: fractional | 10109 | 485473 | 1.57 |
| WindowedGSBModel: fractional + k-core | 10109 | 485473 | 2.54 |

## Fit all 100 queries

Apriori mines termsets with minimum support `MIN_FREQ`. Calling `fit()` without a query subset uses every loaded query and applies the collection's preprocessing.

In [4]:
fit_rows = []
for name, model in models.items():
    start = perf_counter()
    model.fit(min_freq=MIN_FREQ,stopwords=True)
    timings[name]["fit"] = perf_counter() - start
    termsets = sum(len(vector) for vector in model._queryVectors)
    fit_rows.append((name, len(model._query_ids), termsets, f"{timings[name]['fit']:.2f}"))
    print(f"Fitted {name}: {len(model._query_ids)} queries", flush=True)

show_table(("Model", "Queries", "Total mined termsets", "Fit time (s)"), fit_rows)

Fitted SetBasedModel: 100 queries
Fitted GSBModel: 100 queries
Fitted GSBModel + k-core: 100 queries
Fitted WindowedGSBModel: fixed: 100 queries
Fitted WindowedGSBModel: fixed + k-core: 100 queries
Fitted WindowedGSBModel: fractional: 100 queries
Fitted WindowedGSBModel: fractional + k-core: 100 queries


| Model | Queries | Total mined termsets | Fit time (s) |
| --- | --- | --- | --- |
| SetBasedModel | 100 | 8999 | 0.12 |
| GSBModel | 100 | 8999 | 0.13 |
| GSBModel + k-core | 100 | 8999 | 0.13 |
| WindowedGSBModel: fixed | 100 | 8999 | 0.13 |
| WindowedGSBModel: fixed + k-core | 100 | 8999 | 0.13 |
| WindowedGSBModel: fractional | 100 | 8999 | 0.13 |
| WindowedGSBModel: fractional + k-core | 100 | 8999 | 0.12 |

## Evaluate and compare the models

Use each model's existing `evaluate()` on all fitted queries. Its precision and recall values average the values at relevant hits; they are not standard MAP or recall@k. **MRR** is the mean of the stored reciprocal ranks.

Build, fit, and evaluation times below describe this single demo run.

In [5]:
results = {}
evaluation_rows = []
for name, model in models.items():
    start = perf_counter()
    precision, recall = model.evaluate()
    timings[name]["evaluate"] = perf_counter() - start
    results[name] = {
        "mean_precision_at_hits": float(np.mean(precision)),
        "mean_recall_at_hits": float(np.mean(recall)),
        "mrr": float(np.mean(model.reciprocal_ranks)),
    }
    scores = results[name]
    evaluation_rows.append((
        name, len(model.ranking),
        f"{scores['mean_precision_at_hits']:.4f}",
        f"{scores['mean_recall_at_hits']:.4f}",
        f"{scores['mrr']:.4f}",
        f"{timings[name]['evaluate']:.2f}",
    ))

show_table(("Model", "Queries evaluated", "Mean precision at hits", "Mean recall at hits", "MRR", "Evaluate time (s)"), evaluation_rows)

| Model | Queries evaluated | Mean precision at hits | Mean recall at hits | MRR | Evaluate time (s) |
| --- | --- | --- | --- | --- | --- |
| SetBasedModel | 100 | 0.2050 | 0.5237 | 0.6507 | 0.08 |
| GSBModel | 100 | 0.1999 | 0.5237 | 0.6463 | 0.07 |
| GSBModel + k-core | 100 | 0.2004 | 0.5237 | 0.6449 | 0.07 |
| WindowedGSBModel: fixed | 100 | 0.2264 | 0.5237 | 0.7164 | 0.08 |
| WindowedGSBModel: fixed + k-core | 100 | 0.2268 | 0.5237 | 0.7261 | 0.08 |
| WindowedGSBModel: fractional | 100 | 0.2199 | 0.5237 | 0.7016 | 0.08 |
| WindowedGSBModel: fractional + k-core | 100 | 0.2163 | 0.5237 | 0.6949 | 0.08 |

## Inspect a query's retrieved documents

Change `QUERY_ID`, `MODEL_NAME`, and `TOP_K` below, then rerun this cell. All rankings remain available in `models`, and aggregate scores remain in `results`.

In [6]:
QUERY_ID = next(iter(collection.queries))
MODEL_NAME = "WindowedGSBModel: fixed"
TOP_K = 10

model = models[MODEL_NAME]
query_position = model._query_ids.index(QUERY_ID)
judgments = collection.qrels.get(QUERY_ID, {})
documents = {document.id: document for document in collection.docs}

print(f"Query {QUERY_ID}: {collection.queries[QUERY_ID]}")
print(f"Model: {MODEL_NAME}")
print("Tokens:", preprocessor(collection.queries[QUERY_ID]))

ranking_rows = []
for rank, document_id in enumerate(model.ranking[query_position][:TOP_K], start=1):
    title = documents[document_id].text.splitlines()[0]
    grade = judgments.get(str(document_id), 0)
    ranking_rows.append((rank, document_id, "yes" if grade >= 1 else "no", title))
show_table(("Rank", "Document ID", "Relevant", "Title"), ranking_rows)

print("First five document IDs for the same query across all models:")
show_table(("Model", "Top five document IDs"), [
    (name, ", ".join(str(doc_id) for doc_id in candidate.ranking[candidate._query_ids.index(QUERY_ID)][:5]))
    for name, candidate in models.items()
])

Query 1: What are the effects of calcium on the physical properties of mucus from CF patients?
Model: WindowedGSBModel: fixed
Tokens: ['what', 'are', 'the', 'effects', 'of', 'calcium', 'on', 'the', 'physical', 'properties', 'of', 'mucus', 'from', 'cf', 'patients']


| Rank | Document ID | Relevant | Title |
| --- | --- | --- | --- |
| 1 | 533 | yes | Effects of calcium on intestinal mucin: implications for cystic |
| 2 | 437 | no | Pulmonary aspects of cystic fibrosis.  pp.  3-24. |
| 3 | 392 | yes | Recent advances in cystic fibrosis research. |
| 4 | 593 | yes | Physical properties of bronchial secretion. |
| 5 | 875 | yes | Mucous glycoproteins in cystic fibrosis.  pp.  108-21. |
| 6 | 950 | no | Cystic fibrosis: isolation and physical properties of a salivary |
| 7 | 139 | yes | Purification and properties of the calcium-precipitable protein in |
| 8 | 439 | yes | Mucous glycoproteins.  pp.  165-77. |
| 9 | 499 | no | Pathophysiology of mucus secretion in cystic fibrosis. |
| 10 | 441 | yes | Water and electrolytes of exocrine secretions.  pp.  179-91. |

First five document IDs for the same query across all models:


| Model | Top five document IDs |
| --- | --- |
| SetBasedModel | 533, 392, 568, 437, 856 |
| GSBModel | 533, 392, 593, 875, 950 |
| GSBModel + k-core | 533, 392, 593, 875, 950 |
| WindowedGSBModel: fixed | 533, 437, 392, 593, 875 |
| WindowedGSBModel: fixed + k-core | 533, 437, 392, 593, 875 |
| WindowedGSBModel: fractional | 533, 392, 437, 568, 593 |
| WindowedGSBModel: fractional + k-core | 533, 392, 437, 568, 593 |